# 21 — Grid Search sebagai Pembanding Optuna, dan Analisis Selisih Akurasi dengan Penelitian Lain

Notebook ini menjawab dua permintaan bimbingan (19 Okt 2026):

1. **Bagian A — grid search.** Hyperparameter model S3 dicari dengan *grid search*, memakai protokol bersarang yang **sama persis** dengan Optuna di notebook 19. Hasilnya dibandingkan dengan S3 dan S4 (Optuna): apakah cara optimasi mengubah hasil?
2. **Bagian B — analisis.** Mengapa akurasi model usulan lebih rendah daripada beberapa penelitian di Tabel I paper? Dianalisis dari prediksi S4 yang **sudah ada** (tanpa pelatihan ulang): kondisi data, cara pembagian data, kontribusi cabang, kepekaan terhadap hyperparameter, serta ukuran data dan dimensi fitur.

**Skenario S1–S4 tidak berubah.** Grid search di sini adalah pembanding cara optimasi, bukan skenario baru.

| | |
| :--- | :--- |
| **Masukan** | `data/processed/kohort_dcan.csv` (notebook 09) · FC tangent 3 atlas per fold dan FC korelasi HCP (notebook 10/11) · morfometri 360 parcel × 5 (notebook 12/13) · prediksi S3 `result/tabel/27_S3_prediksi.csv` (notebook 18) · prediksi dan hasil Optuna S4 `result/tabel/29_S4_*.csv`, `29_cache_5run/` (notebook 19) |
| **Keluaran** | tabel `result/tabel/31_*.csv` · gambar `result/figure/31_*.png` · ringkasan `result/metrics/S4grid_ringkasan.json` · database grid `result/tabel/31_grid/fold<k>.db` |
| **Waktu** | grid 27 kombinasi × 5 fold ±11 jam + pelatihan akhir ±1,5 jam. Bagian B hanya beberapa detik. Pencarian tersimpan di SQLite, jadi bila dihentikan bisa dilanjutkan dengan menjalankan ulang sel yang sama |

## 1. Persiapan

In [1]:
import gc
import sys
import json
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy import stats
from IPython.display import display
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))
import importlib
import src.usulan
importlib.reload(src.usulan)
import src.optuna_usulan
importlib.reload(src.optuna_usulan)
import optuna
from src.metrics import hitung_metrik
from src.usulan import DEVICE, ModelUsulan, buat_graf, fitur_linear_pca, fitur_simpul_baris, jalankan_cv_usulan, sisi_topk
from src.optuna_usulan import HIPER_S3, GRID_DEFAULT, cari_hiper_grid_fold, pisah_hiper

# ----- Parameter (sama dengan notebook 19, kecuali cara pencarian) -----
SKENARIO = "S4-grid"
ATLAS = {"gordon2014": 352, "hcp2016": 379, "markov2012": 205}
ATLAS_GRAF = "hcp2016"
N_SIMPUL = 360
K_SISI = 20
CFG_LATIH = dict(lr=1e-3, weight_decay=1e-3, batch_size=32, max_epochs=200, min_epochs=20, patience=40, val_size=0.15, seeds=(0, 1, 2, 3, 4))
GRID = dict(GRID_DEFAULT)          # hidden × n_layer × lr; hyperparameter lain tetap = S3
N_KOMBINASI = int(np.prod([len(v) for v in GRID.values()]))

PROC = ROOT / "data" / "processed"
FITUR_22 = PROC / "fitur_dcan_22"
TABEL_DIR = ROOT / "result" / "tabel"
FIG_DIR = ROOT / "result" / "figure"
METRIK_DIR = ROOT / "result" / "metrics"
GRID_DIR = TABEL_DIR / "31_grid"            # satu database SQLite per fold
CACHE_DIR = TABEL_DIR / "31_cache_grid"     # cache pelatihan akhir per fold
for d in (TABEL_DIR, FIG_DIR, METRIK_DIR, GRID_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

kohort = pd.read_csv(PROC / "kohort_dcan.csv")
y = kohort["label"].to_numpy()
fold = kohort["fold_gabungan"].to_numpy()
situs = kohort["situs"].to_numpy()
dataset = kohort["dataset"].to_numpy()
F_SMRI = np.load(PROC / "fitur_smri_dcan" / "fitur_smri_dcan.npz", allow_pickle=True)
print(f"Perangkat: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Kohort: {len(y)} subjek, {y.sum()} ASD, {(1 - y).sum()} kontrol, {len(np.unique(situs))} situs")

Perangkat: cuda (NVIDIA GeForce RTX 3050 6GB Laptop GPU)
Kohort: 1638 subjek, 714 ASD, 924 kontrol, 34 situs


## 2. Graf Otak, Fitur sMRI, dan Fitur Fenotipik (sama dengan S3/S4)

Bagian ini **sama persis** dengan notebook 18–19: graf 360 parcel dengan 20 sisi |korelasi| terbesar per simpul, morfometri 5 fitur per parcel, fenotipik 4 fitur (log usia, jenis kelamin, FIQ yang kosong diisi median data latih, penanda FIQ kosong), dan cabang linear berupa skor PCA FC tangent 3 atlas. Semua statistik fitur dihitung dari data latih setiap fold. Penjelasan dan gambar setiap langkah ada di notebook 18.

In [2]:
t0 = time.perf_counter()
KORELASI = np.load(FITUR_22 / f"korelasi_{ATLAS_GRAF}.npy", mmap_mode="r")
SISI = sisi_topk(KORELASI, ATLAS[ATLAS_GRAF], N_SIMPUL, K_SISI)
print(f"Sisi graf siap untuk {len(SISI)} subjek ({time.perf_counter() - t0:.0f} detik)")


def morfo_relatif(M):
    # (N, R, 5): luas & volume -> proporsi terhadap total subjek; ketebalan, kelengkungan, sulkus apa adanya
    M = M.astype(np.float64).copy()
    M[:, :, 0] /= M[:, :, 0].sum(1, keepdims=True)
    M[:, :, 1] /= M[:, :, 1].sum(1, keepdims=True)
    return M.astype(np.float32)


def standar_latih(X, idx_latih):
    mu = X[idx_latih].mean(0, keepdims=True)
    sd = X[idx_latih].std(0, keepdims=True) + 1e-6
    return ((X - mu) / sd).astype(np.float32)


assert list(F_SMRI["sub"]) == list(kohort["sub"]), "urutan subjek sMRI tidak sama dengan kohort"
MORFO = morfo_relatif(F_SMRI["morfo_hcp2016"])

FIQ_KOSONG = kohort["fiq"].isna().to_numpy()
FEN_MENTAH = np.column_stack([np.log(kohort["usia"].to_numpy(float)), (kohort["jenis_kelamin"] == 2).to_numpy(float),
                              kohort["fiq"].to_numpy(float), FIQ_KOSONG.astype(float)])


def fitur_fenotipik(idx_latih):
    X = FEN_MENTAH.copy()
    X[FIQ_KOSONG, 2] = np.nanmedian(X[idx_latih, 2])
    for j in (0, 2):
        X[:, j] = (X[:, j] - X[idx_latih, j].mean()) / (X[idx_latih, j].std() + 1e-6)
    return X.astype(np.float32)


_simpan = {}


def siapkan_fold(k, idx_latih_penuh):
    # Data PyG untuk SEMUA subjek pada fold k; statistik fitur hanya dari idx_latih_penuh
    if _simpan.get("fold") == k:
        return _simpan["data"]
    _simpan.clear()
    gc.collect()
    V = {a: np.load(FITUR_22 / f"tangent_{a}_gab_{k}.npy", mmap_mode="r") for a in ATLAS}
    X_fc = fitur_simpul_baris(V[ATLAS_GRAF], ATLAS[ATLAS_GRAF], N_SIMPUL, idx_latih_penuh)
    X_simpul = np.concatenate([X_fc, standar_latih(MORFO, idx_latih_penuh)], axis=2)
    del X_fc
    X_linear = fitur_linear_pca([V[a] for a in ATLAS], idx_latih_penuh)
    data = buat_graf(y, X_simpul, SISI, X_linear=X_linear, X_fenotipe=fitur_fenotipik(idx_latih_penuh))
    _simpan.update(fold=k, data=data)
    return data


def buat_model_hiper(contoh, hiper_model):
    return ModelUsulan(N_SIMPUL, d_simpul=contoh.x.size(1), d_linear=contoh.x_lin.size(1), d_fenotipe=contoh.x_fen.size(1), **hiper_model)


t0 = time.perf_counter()
data0 = siapkan_fold(0, np.where(fold != 0)[0])
D_SIMPUL, D_LINEAR, D_FEN = data0[0].x.size(1), data0[0].x_lin.size(1), data0[0].x_fen.size(1)
N_PARAM_S3 = sum(p.numel() for p in buat_model_hiper(data0[0], pisah_hiper(HIPER_S3)[0]).parameters())
display(pd.DataFrame({"Bagian": ["fitur simpul per parcel", "fitur cabang linear (komponen PCA)", "fitur fenotipik", "parameter model (pengaturan S3)"],
                      "Ukuran": [D_SIMPUL, D_LINEAR, D_FEN, N_PARAM_S3]}).style.set_caption(f"Fitur fold 0 ({time.perf_counter() - t0:.0f} detik)"))
del data0

Sisi graf siap untuk 1638 subjek (15 detik)


,Bagian,Ukuran
0,fitur simpul per parcel,384
1,fitur cabang linear (komponen PCA),3927
2,fitur fenotipik,4
3,parameter model (pengaturan S3),430862


## 3. Grid Search per Fold

**Apa yang sama dengan Optuna (notebook 19):** di setiap fold luar, data latih dibagi menjadi 85% latih-dalam dan 15% validasi-tuning (bertingkat situs × diagnosis, potongan yang sama), setiap kombinasi dilatih dengan seed yang sama dan dinilai dengan **AUC validasi-tuning**. Data uji fold tidak pernah dilihat.

**Apa yang berbeda:**

| | Optuna (S4) | Grid search (notebook ini) |
| :--- | :--- | :--- |
| Hyperparameter yang dicari | 11 (dimensi, lapisan, *head*, 5 *dropout*/bobot, *learning rate*, *weight decay*, *batch*) | 3 yang paling berpengaruh: dimensi tersembunyi, jumlah lapisan GATv2, *learning rate*; 8 lainnya tetap = S3 |
| Cara memilih kandidat | TPE (belajar dari trial sebelumnya), 20 trial | semua kombinasi dicoba, 27 trial |
| Penghentian dini trial | ya (*median pruner*) | tidak, setiap kombinasi dilatih penuh |

Kombinasi pengaturan S3 (64, 2 lapisan, 10⁻³) termasuk di dalam grid, sehingga pengaturan terbaik grid minimal sama dengan S3 pada validasi-tuning.

In [3]:
display(pd.DataFrame({"Hyperparameter": list(GRID), "Nilai yang dicoba": [str(v) for v in GRID.values()],
                      "Nilai S3": [HIPER_S3[k] for k in GRID]}).style.set_caption(f"Grid: {N_KOMBINASI} kombinasi per fold"))
display(pd.DataFrame({"Hyperparameter tetap": [k for k in HIPER_S3 if k not in GRID], "Nilai (= S3)": [HIPER_S3[k] for k in HIPER_S3 if k not in GRID]}))

,Hyperparameter,Nilai yang dicoba,Nilai S3
0,hidden,"[32, 64, 128]",64.000000
1,n_layer,"[1, 2, 3]",2.000000
2,lr,"[0.0005, 0.001, 0.002]",0.001000


,Hyperparameter tetap,Nilai (= S3)
0,heads,4.000
1,dropout,0.500
2,dropout_input,0.200
3,dropedge,0.200
4,dropout_linear,0.200
5,bobot_bantu,0.500
6,weight_decay,0.001
7,batch_size,32.000


Sel berikut adalah bagian terlama (±11 jam). Setiap baris log menunjukkan satu kombinasi selesai beserta AUC validasi-tuning-nya.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
STUDY = {}
t0 = time.perf_counter()
for k in [int(v) for v in sorted(np.unique(fold))]:
    latih_penuh = np.where(fold != k)[0]
    data_k = siapkan_fold(k, latih_penuh)
    STUDY[k] = cari_hiper_grid_fold(k, data_k, latih_penuh, y, situs, buat_model_hiper, GRID, GRID_DIR / f"fold{k}.db", cfg_dasar=CFG_LATIH)
    del data_k
    print(f"  fold {k}: AUC val terbaik {STUDY[k].best_value * 100:.2f} {STUDY[k].best_params} | {(time.perf_counter() - t0) / 60:.0f} menit berjalan", flush=True)
_simpan.clear()
gc.collect()
print(f"\nGrid search selesai dalam {(time.perf_counter() - t0) / 60:.1f} menit (≈0 berarti semua kombinasi dibaca dari database).")

    fold 0 kombinasi  1/27: {'hidden': 32, 'n_layer': 3, 'lr': 0.0005} AUC val 73.98 | 3.6 menit | terbaik 73.98
    fold 0 kombinasi  2/27: {'hidden': 128, 'n_layer': 1, 'lr': 0.002} AUC val 80.74 | 4.8 menit | terbaik 80.74
    fold 0 kombinasi  3/27: {'hidden': 64, 'n_layer': 3, 'lr': 0.001} AUC val 77.39 | 5.8 menit | terbaik 80.74
    fold 0 kombinasi  4/27: {'hidden': 64, 'n_layer': 3, 'lr': 0.002} AUC val 78.13 | 5.0 menit | terbaik 80.74
    fold 0 kombinasi  5/27: {'hidden': 32, 'n_layer': 3, 'lr': 0.001} AUC val 75.49 | 2.9 menit | terbaik 80.74
    fold 0 kombinasi  6/27: {'hidden': 64, 'n_layer': 3, 'lr': 0.0005} AUC val 74.81 | 4.6 menit | terbaik 80.74
    fold 0 kombinasi  7/27: {'hidden': 128, 'n_layer': 3, 'lr': 0.001} AUC val 74.69 | 16.0 menit | terbaik 80.74


### 3.1 Hasil grid per fold dan perbandingan dengan Optuna (validasi-tuning)

- **Tabel**: AUC validasi-tuning pengaturan S3, terbaik Optuna, dan terbaik grid, beserta kombinasi grid terbaik.
- **Kiri**: rata-rata AUC validasi-tuning setiap kombinasi (rata-rata 5 fold). Semakin seragam warnanya, semakin kecil pengaruh hyperparameter tersebut.
- **Kanan**: AUC validasi-tuning terbaik per fold.

AUC validasi-tuning ini bersifat **optimis** (nilai terbaik dari banyak percobaan pada potongan ±200 subjek). Perbandingan yang sah ada di data uji (bagian 5).

In [ ]:
semua = pd.concat([st.trials_dataframe().assign(fold=k) for k, st in STUDY.items()])
semua = semua[semua["state"] == "COMPLETE"].copy()
semua["AUC val (%)"] = semua["value"] * 100
semua.to_csv(TABEL_DIR / "31_grid_semua_trial.csv", index=False)
adalah_s3 = (semua["params_hidden"] == HIPER_S3["hidden"]) & (semua["params_n_layer"] == HIPER_S3["n_layer"]) & np.isclose(semua["params_lr"], HIPER_S3["lr"])

OPT = pd.read_csv(TABEL_DIR / "29_S4_optuna_terbaik.csv", dtype={"fold": str}).set_index("fold")
baris = []
for k, st in STUDY.items():
    b = st.best_trial
    baris.append({"fold": k, "AUC val S3 (%)": float(semua.loc[(semua["fold"] == k) & adalah_s3, "AUC val (%)"].iloc[0]),
                  "AUC val terbaik Optuna (%)": float(OPT.loc[str(k), "AUC val terbaik (%)"]),
                  "AUC val terbaik grid (%)": b.value * 100, **{f"grid: {n}": v for n, v in b.params.items()},
                  "menit per kombinasi (rata-rata)": semua.loc[semua["fold"] == k, "user_attrs_menit"].mean()})
tab_grid = pd.DataFrame(baris).set_index("fold")
display(tab_grid.round(3).style.set_caption("Grid search per fold (validasi-tuning)"))
tab_grid.to_csv(TABEL_DIR / "31_grid_terbaik.csv")

HIPER_FOLD = {k: pisah_hiper({**HIPER_S3, **st.best_params})[0] for k, st in STUDY.items()}
CFG_FOLD = {k: pisah_hiper({**HIPER_S3, **st.best_params})[1] for k, st in STUDY.items()}

peta = semua.pivot_table(index=["params_n_layer", "params_lr"], columns="params_hidden", values="AUC val (%)", aggfunc="mean")
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1, 1.2]})
im = axes[0].imshow(peta.to_numpy(), cmap="viridis", aspect="auto")
for i in range(peta.shape[0]):
    for j in range(peta.shape[1]):
        axes[0].text(j, i, f"{peta.iat[i, j]:.1f}", ha="center", va="center", color="white" if peta.iat[i, j] < peta.to_numpy().mean() else "black", fontsize=9)
axes[0].set_xticks(range(peta.shape[1])); axes[0].set_xticklabels([f"dimensi {c}" for c in peta.columns])
axes[0].set_yticks(range(peta.shape[0])); axes[0].set_yticklabels([f"{l} lapisan, lr {r:g}" for l, r in peta.index])
fig.colorbar(im, ax=axes[0], label="AUC validasi-tuning (%), rata-rata 5 fold")
axes[0].set_title("AUC validasi-tuning setiap kombinasi grid", fontweight="bold")
x = np.arange(len(tab_grid))
for j, (kol, wrn) in enumerate([("AUC val S3 (%)", "#A0AEC0"), ("AUC val terbaik Optuna (%)", "#2B6CB0"), ("AUC val terbaik grid (%)", "#DD6B20")]):
    axes[1].bar(x + (j - 1) * 0.27, tab_grid[kol], 0.27, color=wrn, label=kol.replace(" (%)", ""))
axes[1].set_xticks(x); axes[1].set_xticklabels([f"fold {k}" for k in tab_grid.index]); axes[1].set_ylim(60, 90)
axes[1].set_ylabel("AUC validasi-tuning (%)"); axes[1].legend(fontsize=8); axes[1].grid(axis="y", alpha=0.4, ls="--")
axes[1].set_title("S3 vs terbaik Optuna vs terbaik grid", fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "31_1_grid_validasi.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Pelatihan Akhir 5-Fold dengan Pengaturan Terbaik Grid

Sama dengan notebook 19: setiap fold dilatih ulang dengan pengaturan terbaiknya memakai **5 seed (5 run)**, dengan data latih, potongan validasi untuk *early stopping*, dan seed yang sama dengan S3 dan S4. Karena itu run ke-r grid bisa dibandingkan berpasangan dengan run ke-r S3 dan S4.

In [ ]:
def buat_model(contoh, k):
    return buat_model_hiper(contoh, HIPER_FOLD[k])

t0 = time.perf_counter()
HASIL = jalankan_cv_usulan(y, situs, fold, siapkan_fold, buat_model, cfg=CFG_LATIH, cache_dir=CACHE_DIR, cfg_fold=CFG_FOLD)
_simpan.clear()
print(f"\nSelesai dalam {(time.perf_counter() - t0) / 60:.1f} menit (≈0 berarti semua fold dibaca dari cache).")

## 5. Hasil Bagian A: Grid Search vs Optuna vs S3 vs Baseline

- **Per run**: rata-rata ± SD antar 5 run (setiap run = rata-rata 5 fold), sama dengan cara pelaporan S1–S4.
- **Ensemble**: probabilitas 5 run dirata-rata, lalu dihitung rata-rata 5 fold.
- **Uji t berpasangan** membandingkan run ke-r antar metode (5 pasangan); **uji McNemar** membandingkan prediksi ensemble per subjek (koreksi Holm).

In [ ]:
KOLOM = {"accuracy": "Akurasi", "balanced": "Balanced acc.", "sensitivity": "Sensitivitas", "specificity": "Spesifisitas",
         "precision": "Presisi", "f1": "F1", "auc": "AUC"}


def metrik_per_fold(p):
    df = pd.DataFrame([{"fold": k, **hitung_metrik(y[fold == k], p[fold == k])} for k in sorted(np.unique(fold))])
    df["balanced"] = (df["sensitivity"] + df["specificity"]) / 2
    return df


def rata_fold(p):
    return metrik_per_fold(p)[list(KOLOM)].mean() * 100


S3 = pd.read_csv(TABEL_DIR / "27_S3_prediksi.csv")
S4 = pd.read_csv(TABEL_DIR / "29_S4_prediksi.csv")
for nama, df in (("S3", S3), ("S4", S4)):
    assert (df["FILE_ID"].to_numpy() == kohort["FILE_ID"].to_numpy()).all(), f"urutan subjek {nama} berbeda"
N_RUN = HASIL["p_run"].shape[1]
kol_run = [f"p_run{r + 1}" for r in range(N_RUN)]
RUN = {"S3 (hyperparameter tetap)": S3[kol_run].to_numpy(), "S4 Optuna": S4[kol_run].to_numpy(), "S4 grid search": HASIL["p_run"]}
ENS = {"S3 (hyperparameter tetap)": S3["p_ensemble"].to_numpy(), "S4 Optuna": S4["p_ensemble"].to_numpy(),
       "S4 grid search": HASIL["p"], "Baseline LR": S4["p_lr"].to_numpy()}
PER_RUN = {n: pd.DataFrame([rata_fold(P[:, r]) for r in range(N_RUN)]) for n, P in RUN.items()}

baris = []
for n, p in ENS.items():
    e = rata_fold(p)
    r = PER_RUN.get(n)
    baris.append({"Model": n,
                  "Akurasi per run (%)": f"{r['accuracy'].mean():.2f} ± {r['accuracy'].std(ddof=1):.2f}" if r is not None else "–",
                  "AUC per run (%)": f"{r['auc'].mean():.2f} ± {r['auc'].std(ddof=1):.2f}" if r is not None else "–",
                  **{f"{v} ensemble (%)": round(e[k], 2) for k, v in KOLOM.items()}})
tab_banding = pd.DataFrame(baris).set_index("Model")
display(tab_banding.style.set_caption("Grid search vs Optuna vs S3 vs baseline (1.638 subjek, fold sama)"))
tab_banding.to_csv(TABEL_DIR / "31_grid_vs_optuna_ringkasan.csv")

pasangan = [("S4 grid search", "S4 Optuna"), ("S4 grid search", "S3 (hyperparameter tetap)"), ("S4 Optuna", "S3 (hyperparameter tetap)")]
uji_t = pd.DataFrame([{"Perbandingan": f"{a} − {b}", "Selisih akurasi (poin)": (PER_RUN[a]["accuracy"] - PER_RUN[b]["accuracy"]).mean(),
                       "p akurasi": stats.ttest_rel(PER_RUN[a]["accuracy"], PER_RUN[b]["accuracy"]).pvalue,
                       "Selisih AUC (poin)": (PER_RUN[a]["auc"] - PER_RUN[b]["auc"]).mean(),
                       "p AUC": stats.ttest_rel(PER_RUN[a]["auc"], PER_RUN[b]["auc"]).pvalue} for a, b in pasangan])
display(uji_t.round(4).style.set_caption("Uji t berpasangan antar 5 run"))
uji_t.to_csv(TABEL_DIR / "31_grid_uji_t.csv", index=False)


def mcnemar(pa, pb):
    A, B = (pa >= 0.5).astype(int) == y, (pb >= 0.5).astype(int) == y
    n01, n10 = int((~A & B).sum()), int((A & ~B).sum())
    return {"Selisih akurasi (poin)": round((B.mean() - A.mean()) * 100, 2), "Hanya benar di kedua": n01, "Hanya benar di pertama": n10,
            "p": stats.binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0}


uji_m = pd.DataFrame([{"Pertama → kedua": f"{b} → {a}", **mcnemar(ENS[b], ENS[a])}
                      for a, b in [("S4 grid search", "S4 Optuna"), ("S4 grid search", "S3 (hyperparameter tetap)"), ("S4 grid search", "Baseline LR")]])
urut = np.argsort(uji_m["p"].to_numpy())
uji_m.loc[uji_m.index[urut], "p Holm"] = np.minimum(1, np.maximum.accumulate(uji_m["p"].to_numpy()[urut] * (len(uji_m) - np.arange(len(uji_m)))))
uji_m["Kesimpulan"] = np.where(uji_m["p Holm"] >= 0.05, "setara (tidak berbeda bermakna)",
                               np.where(uji_m["Selisih akurasi (poin)"] > 0, "kedua lebih baik secara bermakna", "kedua lebih rendah secara bermakna"))
display(uji_m.round(4).style.set_caption("Uji McNemar ensemble + koreksi Holm"))
uji_m.to_csv(TABEL_DIR / "31_grid_uji_mcnemar.csv", index=False)

pd.DataFrame({"FILE_ID": kohort["FILE_ID"], "dataset": dataset, "situs": situs, "fold": fold, "y_true": y, "p_ensemble": HASIL["p"],
              **{f"p_run{r + 1}": HASIL["p_run"][:, r] for r in range(N_RUN)}, "p_graf": HASIL["p_graf"], "p_linear": HASIL["p_linear"],
              "p_fenotipik": HASIL["p_fenotipik"]}).to_csv(TABEL_DIR / "31_S4grid_prediksi.csv", index=False)
with open(METRIK_DIR / "S4grid_ringkasan.json", "w", encoding="utf-8") as f:
    json.dump({"skenario": SKENARIO, "notebook": "21", "n_subjek": int(len(y)), "n_run": int(N_RUN), "grid": GRID,
               "rata_rata_antar_run": PER_RUN["S4 grid search"].mean().round(4).to_dict(),
               "sd_antar_run": PER_RUN["S4 grid search"].std(ddof=1).round(4).to_dict(),
               "ensemble": rata_fold(HASIL["p"]).round(4).to_dict(),
               "hiper_model_per_fold": {int(k): v for k, v in HIPER_FOLD.items()}, "latih_per_fold": {int(k): v for k, v in CFG_FOLD.items()},
               "cfg_latih": {k: list(v) if isinstance(v, tuple) else v for k, v in CFG_LATIH.items()}}, f, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
WARNA = {"S3 (hyperparameter tetap)": "#A0AEC0", "S4 Optuna": "#2B6CB0", "S4 grid search": "#DD6B20"}
for ax, (m, judul) in zip(axes, [("accuracy", "Akurasi"), ("auc", "AUC")]):
    for j, (n, df) in enumerate(PER_RUN.items()):
        ax.bar(j, df[m].mean(), 0.6, color=WARNA[n])
        ax.scatter(np.full(N_RUN, j), df[m], color="black", s=14, zorder=3)
        ax.text(j, df[m].max() + 0.25, f"{df[m].mean():.2f}", ha="center", fontsize=9)
    ax.axhline(rata_fold(ENS["Baseline LR"])[m], color="#C53030", ls="--", lw=1.2, label="Baseline LR")
    ax.set_xticks(range(len(PER_RUN))); ax.set_xticklabels(list(PER_RUN), fontsize=9)
    nilai = np.concatenate([df[m].to_numpy() for df in PER_RUN.values()])
    ax.set_ylim(nilai.min() - 2, max(nilai.max(), rata_fold(ENS["Baseline LR"])[m]) + 1.5)
    ax.set_ylabel(f"{judul} (%)"); ax.legend(fontsize=8, loc="lower right"); ax.grid(axis="y", alpha=0.4, ls="--")
    ax.set_title(f"{judul} per run (batang = rata-rata 5 run, titik = setiap run)", fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "31_2_grid_vs_optuna.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Bagian B: Mengapa Akurasi Lebih Rendah daripada Beberapa Penelitian Lain?

Bagian ini memakai **prediksi S4 (metode usulan) dan baseline LR yang sudah ada** dari notebook 19, tanpa pelatihan ulang. Ada lima sudut pandang:

| Sub-bagian | Pertanyaan |
| :--- | :--- |
| 6.1 | Apakah kondisi data (dataset, jenis kelamin, situs) membuat akurasi tampak lebih tinggi atau lebih rendah? |
| 6.2 | Seberapa besar akurasi bisa berubah hanya karena satu kali pembagian 80/20? |
| 6.3 | Cabang mana yang membawa sinyal, dan seberapa kuat cabang graf? |
| 6.4 | Apakah hyperparameter yang membatasi akurasi? |
| 6.5 | Bagaimana perbandingan jumlah subjek dengan jumlah fitur dan parameter? |

> **Batasan analisis 6.1–6.2:** model tetap dilatih pada seluruh kohort; yang diubah hanya kelompok subjek uji yang dihitung. Ini **tidak sama** dengan melatih ulang model hanya pada subkelompok tersebut, sehingga angkanya menunjukkan arah pengaruh kondisi data, bukan angka yang pasti akan diperoleh penelitian lain.

### 6.1 Akurasi pada subkelompok yang meniru kondisi penelitian lain

Metrik di sini dihitung dari **semua subjek uji dalam subkelompok sekaligus** (bukan rata-rata 5 fold), sehingga baris "Semua subjek" bisa sedikit berbeda dari angka utama 73,68%.

In [ ]:
P_S4 = S4["p_ensemble"].to_numpy()
P_LR = S4["p_lr"].to_numpy()
usia = kohort["usia"].to_numpy()
laki = kohort["jenis_kelamin"].to_numpy() == 1
n_situs = pd.Series(situs).value_counts()
situs_terbesar = n_situs.index[0]


def ringkas_sub(nama, m, kondisi):
    a, b = hitung_metrik(y[m], P_S4[m]), hitung_metrik(y[m], P_LR[m])
    return {"Subkelompok": nama, "Meniru kondisi": kondisi, "Subjek": int(m.sum()), "Situs": int(pd.Series(situs[m]).nunique()),
            "ASD (%)": y[m].mean() * 100, "Akurasi S4 (%)": a["accuracy"] * 100,
            "Balanced acc. S4 (%)": (a["sensitivity"] + a["specificity"]) / 2 * 100, "AUC S4 (%)": a["auc"] * 100,
            "Akurasi LR (%)": b["accuracy"] * 100, "AUC LR (%)": b["auc"] * 100}


SUB = [("Semua subjek", np.ones(len(y), bool), "kohort penelitian ini"),
       ("Hanya ABIDE I", dataset == "abide1", "Yang, Jha, Dong, Wang, Tang (ABIDE I saja)"),
       ("Hanya ABIDE II", dataset == "abide2", "pembanding ABIDE I"),
       ("Laki-laki", laki, "subjek laki-laki saja"),
       ("Laki-laki, usia 5–40 tahun", laki & (usia >= 5) & (usia <= 40), "Saponaro (laki-laki, 5–40 tahun)"),
       (f"Situs terbesar ({situs_terbesar})", situs == situs_terbesar, "satu situs saja")]
tab_sub = pd.DataFrame([ringkas_sub(*s) for s in SUB]).set_index("Subkelompok")
display(tab_sub.round(2).style.set_caption("S4 (ensemble) dan LR pada subkelompok"))
tab_sub.round(4).to_csv(TABEL_DIR / "31_analisis_subkelompok.csv")

benar = ((P_S4 >= 0.5).astype(int) == y)
per_situs = pd.DataFrame({"situs": situs, "benar": benar}).groupby("situs").agg(n=("benar", "size"), akurasi=("benar", "mean"))
per_situs["akurasi"] *= 100
cukup = per_situs[per_situs["n"] >= 20]
display(pd.DataFrame({"Situs (n ≥ 20)": [len(cukup)], "Akurasi terendah (%)": [cukup["akurasi"].min()], "Median (%)": [cukup["akurasi"].median()],
                      "Tertinggi (%)": [cukup["akurasi"].max()], "Situs tertinggi": [cukup["akurasi"].idxmax()], "Situs terendah": [cukup["akurasi"].idxmin()]})
        .round(2).style.set_caption("Sebaran akurasi S4 antar situs"))

### 6.2 Pengaruh satu kali pembagian data 80/20

Beberapa penelitian melaporkan hasil dari **satu kali** pembagian latih/uji. Untuk melihat seberapa besar akurasi bisa berubah hanya karena subjek mana yang masuk data uji, diambil 2.000 kali sampel acak 20% subjek (proporsi ASD/kontrol dijaga) dari prediksi uji S4, lalu akurasinya dihitung. Akurasi kelima fold asli ditampilkan sebagai pembanding.

In [ ]:
rng = np.random.default_rng(0)
i_asd, i_td = np.where(y == 1)[0], np.where(y == 0)[0]
n_asd, n_td = round(0.2 * len(i_asd)), round(0.2 * len(i_td))
B = 2000
acc_s4, acc_lr = np.empty(B), np.empty(B)
for b in range(B):
    m = np.concatenate([rng.choice(i_asd, n_asd, replace=False), rng.choice(i_td, n_td, replace=False)])
    acc_s4[b] = ((P_S4[m] >= 0.5).astype(int) == y[m]).mean() * 100
    acc_lr[b] = ((P_LR[m] >= 0.5).astype(int) == y[m]).mean() * 100
fold_s4 = metrik_per_fold(P_S4)["accuracy"].to_numpy() * 100
tab_split = pd.DataFrame({"Model": ["S4 (ensemble)", "Baseline LR"], "Rata-rata (%)": [acc_s4.mean(), acc_lr.mean()],
                          "SD (%)": [acc_s4.std(), acc_lr.std()], "Persentil 2,5 (%)": [np.percentile(acc_s4, 2.5), np.percentile(acc_lr, 2.5)],
                          "Persentil 97,5 (%)": [np.percentile(acc_s4, 97.5), np.percentile(acc_lr, 97.5)],
                          "Maksimum (%)": [acc_s4.max(), acc_lr.max()]}).set_index("Model")
display(tab_split.round(2).style.set_caption(f"Akurasi pada 2.000 data uji acak 20% ({n_asd + n_td} subjek)"))
tab_split.round(4).to_csv(TABEL_DIR / "31_analisis_satu_split.csv")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(acc_s4, bins=40, color="#2B6CB0", alpha=0.75, label="S4: data uji acak 20%")
for i, a in enumerate(fold_s4):
    ax.axvline(a, color="#DD6B20", lw=1.4, ls="--", label="S4: 5 fold asli" if i == 0 else None)
ax.axvline(np.percentile(acc_s4, 2.5), color="black", lw=0.8, ls=":")
ax.axvline(np.percentile(acc_s4, 97.5), color="black", lw=0.8, ls=":", label="rentang 95%")
ax.set_xlabel("akurasi (%)"); ax.set_ylabel("jumlah sampel"); ax.legend(fontsize=9); ax.grid(alpha=0.4, ls="--")
ax.set_title("Sebaran akurasi S4 bila hanya satu kali pembagian 80/20", fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "31_3_sebaran_satu_split.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.3 Kontribusi setiap cabang

Akurasi kepala bantu setiap cabang (prediksi satu cabang pada model S4 yang sama, ensemble 5 run) dan bobot fusi β rata-rata. Bila satu cabang sudah menyamai model penuh, berarti sinyal utamanya berasal dari cabang tersebut.

In [ ]:
beta = np.zeros((len(y), 3))
for k in sorted(np.unique(fold)):
    z = np.load(TABEL_DIR / "29_cache_5run" / f"fold{k}.npz", allow_pickle=True)
    beta[z["idx"]] = z["beta"]
CABANG = {"Cabang graf (GATv2 + pooling)": ("p_graf", 0), "Cabang linear (FC tangent PCA)": ("p_linear", 1), "Cabang fenotipik": ("p_fenotipik", 2)}
baris = [{"Bagian": n, "Akurasi (%)": rata_fold(S4[kol].to_numpy())["accuracy"], "AUC (%)": rata_fold(S4[kol].to_numpy())["auc"],
          "β rata-rata": beta[:, j].mean(), "β SD": beta[:, j].std()} for n, (kol, j) in CABANG.items()]
baris += [{"Bagian": "Model penuh S4 (ensemble)", "Akurasi (%)": rata_fold(P_S4)["accuracy"], "AUC (%)": rata_fold(P_S4)["auc"]},
          {"Bagian": "Baseline LR", "Akurasi (%)": rata_fold(P_LR)["accuracy"], "AUC (%)": rata_fold(P_LR)["auc"]}]
tab_cabang = pd.DataFrame(baris).set_index("Bagian")
display(tab_cabang.round(3).style.format(precision=3, na_rep="–").set_caption("Kontribusi cabang pada S4 (rata-rata 5 fold)"))
tab_cabang.round(4).to_csv(TABEL_DIR / "31_analisis_cabang.csv")

### 6.4 Apakah hyperparameter yang membatasi akurasi?

Dibandingkan dua hal: (1) **rentang AUC validasi-tuning** antar semua trial Optuna dan semua kombinasi grid per fold, dan (2) **akurasi uji** per fold untuk S3 (hyperparameter tetap), S4 Optuna, dan S4 grid. Bila AUC validasi berubah lebar antar pengaturan tetapi akurasi uji hampir sama, berarti pilihan hyperparameter bukan pembatas utama.

In [ ]:
opt_semua = pd.read_csv(TABEL_DIR / "29_S4_optuna_semua_trial.csv")
opt_semua = opt_semua[opt_semua["state"] == "COMPLETE"]
uji_fold = {n: metrik_per_fold(p)["accuracy"].to_numpy() * 100 for n, p in ENS.items() if n != "Baseline LR"}
baris = []
for k in sorted(np.unique(fold)):
    vo, vg = opt_semua.loc[opt_semua["fold"] == k, "value"] * 100, semua.loc[semua["fold"] == k, "AUC val (%)"]
    baris.append({"fold": k, "AUC val Optuna min–maks (%)": f"{vo.min():.1f}–{vo.max():.1f}", "rentang Optuna (poin)": vo.max() - vo.min(),
                  "AUC val grid min–maks (%)": f"{vg.min():.1f}–{vg.max():.1f}", "rentang grid (poin)": vg.max() - vg.min(),
                  **{f"akurasi uji {n} (%)": v[k] for n, v in uji_fold.items()}})
tab_hiper = pd.DataFrame(baris).set_index("fold")
display(tab_hiper.round(2).style.set_caption("Kepekaan terhadap hyperparameter: validasi-tuning vs data uji"))
tab_hiper.round(4).to_csv(TABEL_DIR / "31_analisis_kepekaan_hiper.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
folds = sorted(np.unique(fold))
axes[0].boxplot([opt_semua.loc[opt_semua["fold"] == k, "value"] * 100 for k in folds], positions=np.arange(len(folds)) - 0.18, widths=0.3,
                patch_artist=True, boxprops=dict(facecolor="#BEE3F8"))
axes[0].boxplot([semua.loc[semua["fold"] == k, "AUC val (%)"] for k in folds], positions=np.arange(len(folds)) + 0.18, widths=0.3,
                patch_artist=True, boxprops=dict(facecolor="#FBD38D"))
axes[0].set_xticks(range(len(folds))); axes[0].set_xticklabels([f"fold {k}" for k in folds])
axes[0].set_ylabel("AUC validasi-tuning (%)"); axes[0].grid(axis="y", alpha=0.4, ls="--")
axes[0].set_title("AUC validasi-tuning semua pengaturan (biru = Optuna, oranye = grid)", fontweight="bold")
for n, v in uji_fold.items():
    axes[1].plot(folds, v, "-o", lw=1.6, label=n, color=WARNA[n])
axes[1].set_xticks(folds); axes[1].set_xticklabels([f"fold {k}" for k in folds])
axes[1].set_ylabel("akurasi uji (%)"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.4, ls="--")
axes[1].set_title("Akurasi uji per fold (ensemble)", fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "31_4_kepekaan_hiper.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.5 Jumlah subjek dibanding jumlah fitur dan parameter

In [ ]:
n_latih = np.array([(fold != k).sum() for k in sorted(np.unique(fold))])
n_fc = sum(n * (n - 1) // 2 for n in ATLAS.values())
tab_ukuran = pd.DataFrame({"Besaran": ["subjek latih per fold (rata-rata)", "fitur FC tangent 3 atlas", "komponen PCA cabang linear",
                                       "fitur simpul per graf (360 × fitur per parcel)", "parameter model (pengaturan S3)",
                                       "fitur FC per subjek latih", "parameter per subjek latih"],
                           "Nilai": [n_latih.mean(), n_fc, D_LINEAR, N_SIMPUL * D_SIMPUL, N_PARAM_S3, n_fc / n_latih.mean(), N_PARAM_S3 / n_latih.mean()]})
display(tab_ukuran.round(1).style.set_caption("Ukuran data vs dimensi"))
tab_ukuran.round(2).to_csv(TABEL_DIR / "31_analisis_ukuran_data.csv", index=False)

## 7. Cara Membaca Hasil

| Pertanyaan | Dilihat dari |
| :--- | :--- |
| Apakah grid search menemukan pengaturan lebih baik daripada S3 di validasi-tuning? | tabel 3.1 (AUC val grid vs S3) |
| Hyperparameter mana yang paling berpengaruh? | peta warna 3.1: perbedaan warna antar baris (lapisan, *learning rate*) atau antar kolom (dimensi) |
| **Apakah grid search lebih baik, sama, atau lebih rendah daripada Optuna di data uji?** | tabel bagian 5, uji t berpasangan dan McNemar (kolom **p Holm**) |
| Apakah kondisi data menjelaskan selisih dengan penelitian lain? | tabel 6.1: bandingkan akurasi subkelompok dengan "Semua subjek" |
| Seberapa besar angka bisa berubah karena satu kali pembagian 80/20? | tabel dan histogram 6.2 (rentang 95% dan maksimum) |
| Apakah cabang graf menambah sinyal di luar FC linear? | tabel 6.3 (akurasi cabang graf vs linear vs model penuh) |
| Apakah hyperparameter pembatas utamanya? | tabel dan grafik 6.4: rentang AUC validasi lebar tetapi akurasi uji per fold berdekatan berarti **bukan** |
| Seberapa sedikit data dibanding dimensi? | tabel 6.5 |

Bagian ini hanya menyajikan angka. Kesimpulan untuk paper (Bagian *Discussion*) ditulis setelah notebook dijalankan, berdasarkan angka yang benar-benar keluar.